# 03 · Connect the image encoder to a small language model

*Vision-Language Models (VLMs)*, notebook 3 of 4

A contrastive model can rank captions, but it cannot answer a question. Generative vision-language models such as LLaVA add a **connector**: a small trainable module that turns the vision encoder's output into vectors the language model reads as if they were word embeddings. The language model then continues the text, and its answer is supervised like any next-token prediction, except that only the answer counts in the loss. In this notebook you build that path around the encoder you aligned in notebook 2, train it to answer questions about the bar, and compare training with the image encoder frozen and unfrozen.

## What you will build

- `project_visual_tokens`: the linear projector from the vision width to the language model's width;
- `build_input_sequence`: visual tokens followed by the embedded question and answer, plus positions;
- `answer_loss_mask`: which positions the loss supervises, so that only the answer and its end token are trained;
- `read_answers`: the model's answer, read from the right position;
- a visual question answering model trained twice, with the vision encoder frozen and unfrozen, plus an image-swap control.

## Learning objectives

By the end of this notebook you will be able to:

1. explain what a connector does and implement a linear projector between two widths;
2. lay out a multimodal sequence (visual tokens, question, answer, end) and derive its next-token targets;
3. write an answer-only loss mask and check which positions it supervises, including for questions of different lengths;
4. train visual question answering with the vision encoder frozen and unfrozen, and verify which parameters changed;
5. test whether answers depend on the image with an image-swap control, and compare one image-embedding token with sixteen patch tokens.

### How to work through this notebook

Run the cells from top to bottom. When you reach a **✍️ Your turn** cell, type the function yourself and then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook quietly uses a hidden reference version, so you can always keep going and come back later.

Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck.

## Dependencies

- **Tier 1:** NumPy and Matplotlib only, so the notebook runs in JupyterLite in your browser, on the course JupyterHub server and in Google Colab.
- Nothing is downloaded. The vision encoder is retrained from scratch in a few tenths of a second with the code from notebook 2, so this notebook does not depend on files saved earlier.
- Training uses the tiny tensor engine from Base Model Training, defined in a hidden recap cell.
- Every random number comes from a seeded generator. The four training runs together take a few seconds.

## Setup

Run this cell first. It imports NumPy, `time` and Matplotlib and defines `practice`, the helper that checks each function you write.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import time

import matplotlib.pyplot as plt

## 📦 Recap: the tiny tensor engine you built in Base Model Training

The hidden cell below defines the reverse-mode autograd engine from Base Model Training, unchanged: `Tensor` (a NumPy array that remembers how it was computed, so that `backward()` can fill in `.grad`), `parameter`, `zeros_parameter`, `zero_grad`, `cross_entropy`, `layer_norm`, `numerical_grad` and `relative_error`. Every operation in it is gradient-checked. Run the cell; open it only if you want to look back at the code.

In [ ]:
# ── Tiny tensor autograd (NumPy) ── the engine you built in Base Model Training ──
import numpy as np


def _unbroadcast(grad, shape):
    """Sum a gradient back down to `shape`, undoing NumPy broadcasting."""
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(shape):
        if size == 1 and grad.shape[axis] != 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad


class Tensor:
    """A NumPy array that remembers how it was made, so gradients can flow back."""

    def __init__(self, data, requires_grad=False, parents=(), backward=None, op=""):
        self.data = np.asarray(data, dtype=np.float64)
        self.requires_grad = requires_grad
        self.grad = None
        self._parents = parents
        self._backward = backward  # grad_out -> one gradient per parent
        self.op = op

    shape = property(lambda self: self.data.shape)
    ndim = property(lambda self: self.data.ndim)
    T = property(lambda self: self.transpose())

    def __repr__(self):
        return f"Tensor(shape={self.shape}, op={self.op or 'leaf'}, requires_grad={self.requires_grad})"

    @staticmethod
    def lift(value):
        return value if isinstance(value, Tensor) else Tensor(value)

    def _result(self, data, parents, backward, op):
        if any(parent.requires_grad for parent in parents):
            return Tensor(data, True, parents, backward, op)
        return Tensor(data, op=op)

    # arithmetic, with broadcasting
    def __add__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data + other.data, (self, other), lambda g: (g, g), "add")

    __radd__ = __add__

    def __neg__(self):
        return self._result(-self.data, (self,), lambda g: (-g,), "neg")

    def __sub__(self, other):
        return self + (-Tensor.lift(other))

    def __rsub__(self, other):
        return Tensor.lift(other) + (-self)

    def __mul__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data * other.data, (self, other),
                            lambda g: (g * other.data, g * self.data), "mul")

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data / other.data, (self, other),
                            lambda g: (g / other.data, -g * self.data / other.data ** 2), "div")

    def __rtruediv__(self, other):
        return Tensor.lift(other) / self

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float)), "only constant exponents"
        return self._result(self.data ** exponent, (self,),
                            lambda g: (g * exponent * self.data ** (exponent - 1),), "pow")

    def __matmul__(self, other):
        other = Tensor.lift(other)

        def backward(g):
            return (g @ np.swapaxes(other.data, -1, -2), np.swapaxes(self.data, -1, -2) @ g)

        return self._result(self.data @ other.data, (self, other), backward, "matmul")

    # elementwise functions
    def exp(self):
        out = np.exp(self.data)
        return self._result(out, (self,), lambda g: (g * out,), "exp")

    def log(self):
        return self._result(np.log(self.data), (self,), lambda g: (g / self.data,), "log")

    def tanh(self):
        out = np.tanh(self.data)
        return self._result(out, (self,), lambda g: (g * (1 - out ** 2),), "tanh")

    def relu(self):
        return self._result(np.maximum(self.data, 0), (self,), lambda g: (g * (self.data > 0),), "relu")

    def sigmoid(self):
        out = 0.5 * (1 + np.tanh(0.5 * self.data))  # stable for large |x|
        return self._result(out, (self,), lambda g: (g * out * (1 - out),), "sigmoid")

    # reductions and reshaping
    def sum(self, axis=None, keepdims=False):
        def backward(g):
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis)
            return (np.broadcast_to(g, self.shape).copy(),)

        return self._result(self.data.sum(axis=axis, keepdims=keepdims), (self,), backward, "sum")

    def mean(self, axis=None, keepdims=False):
        count = self.data.size if axis is None else np.prod([self.shape[a] for a in np.atleast_1d(axis)])
        return self.sum(axis=axis, keepdims=keepdims) / count

    def reshape(self, *shape):
        return self._result(self.data.reshape(*shape), (self,), lambda g: (g.reshape(self.shape),), "reshape")

    def transpose(self, *axes):
        axes = axes or tuple(reversed(range(self.ndim)))
        inverse = np.argsort(axes)
        return self._result(self.data.transpose(axes), (self,), lambda g: (g.transpose(inverse),), "transpose")

    def __getitem__(self, index):
        def backward(g):
            full = np.zeros_like(self.data)
            np.add.at(full, index, g)  # repeated indices (an embedding row used twice) add up
            return (full,)

        return self._result(self.data[index], (self,), backward, "index")

    def log_softmax(self, axis=-1):
        shifted = self.data - self.data.max(axis=axis, keepdims=True)
        out = shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))
        probs = np.exp(out)
        return self._result(out, (self,), lambda g: (g - probs * g.sum(axis=axis, keepdims=True),), "log_softmax")

    def softmax(self, axis=-1):
        shifted = np.exp(self.data - self.data.max(axis=axis, keepdims=True))
        out = shifted / shifted.sum(axis=axis, keepdims=True)
        return self._result(out, (self,), lambda g: (out * (g - (g * out).sum(axis=axis, keepdims=True)),), "softmax")

    # reverse-mode differentiation
    def backward(self, grad=None):
        order, seen = [], set()

        def visit(node):
            if id(node) not in seen:
                seen.add(id(node))
                for parent in node._parents:
                    visit(parent)
                order.append(node)

        visit(self)
        self.grad = np.ones_like(self.data) if grad is None else np.asarray(grad, dtype=np.float64)
        for node in reversed(order):
            if node._backward is None or node.grad is None:
                continue
            for parent, parent_grad in zip(node._parents, node._backward(node.grad)):
                if not parent.requires_grad or parent_grad is None:
                    continue
                parent_grad = _unbroadcast(np.asarray(parent_grad, dtype=np.float64), parent.shape)
                parent.grad = parent_grad if parent.grad is None else parent.grad + parent_grad


def parameter(*shape, rng, scale=None):
    """A trainable weight, initialised with N(0, scale^2); default scale is He's sqrt(2 / fan_in)."""
    scale = np.sqrt(2.0 / shape[0]) if scale is None else scale
    return Tensor(rng.normal(0.0, scale, shape), requires_grad=True)


def zeros_parameter(*shape):
    return Tensor(np.zeros(shape), requires_grad=True)


def zero_grad(params):
    for p in params:
        p.grad = None


def cross_entropy(logits, targets, mask=None):
    """Mean next-token/class cross-entropy. logits: (..., classes); targets: int (...); mask: 1.0 keeps a position."""
    flat = logits.reshape(-1, logits.shape[-1]).log_softmax(axis=-1)
    targets = np.asarray(targets).reshape(-1)
    picked = flat[np.arange(targets.size), targets]
    if mask is None:
        return -picked.mean()
    weights = np.asarray(mask, dtype=np.float64).reshape(-1)
    return -(picked * weights).sum() / max(weights.sum(), 1.0)


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    centred = x - mean
    var = (centred ** 2).mean(axis=-1, keepdims=True)
    return centred / (var + eps) ** 0.5 * gamma + beta


def numerical_grad(f, tensor, eps=1e-6):
    """Central-difference gradient of scalar f() with respect to tensor.data."""
    grad = np.zeros_like(tensor.data)
    it = np.nditer(tensor.data, flags=["multi_index"])
    for _ in it:
        index = it.multi_index
        original = tensor.data[index]
        tensor.data[index] = original + eps
        plus = float(f().data)
        tensor.data[index] = original - eps
        minus = float(f().data)
        tensor.data[index] = original
        grad[index] = (plus - minus) / (2 * eps)
    return grad


def relative_error(a, b):
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

## 📦 Recap: what you built earlier

The hidden cell below re-defines the pieces from notebooks 1 and 2 under their usual names: `practice_tensor`, the scene generator and the three sets of scenes and images, `channel_stats`, `normalise_images`, `patchify`, `unpatchify`, `prepare`, the image encoder (`init_image_encoder`, `embed_patches`, `encode_images`), `Adam`, the text side (`VOCAB`, `tokenise`, `encode_text`, `l2_normalise`), the losses (`clip_loss`, `positive_mask`, `multi_positive_loss`), `recall_at_k`, and the alignment model with `train_alignment` and `retrieval_report`. Run it; open it only if you want to look back at the code.

In [ ]:
def _fresh(value):
    """A copy of every Tensor inside value, so each check starts from clean gradients."""
    if isinstance(value, Tensor):
        return Tensor(value.data.copy(), requires_grad=value.requires_grad)
    if isinstance(value, dict):
        return {key: _fresh(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return type(value)(_fresh(item) for item in value)
    return value


def _trainable_leaves(value):
    if isinstance(value, Tensor):
        return [value] if value.requires_grad else []
    if isinstance(value, dict):
        return [leaf for key in sorted(value) for leaf in _trainable_leaves(value[key])]
    if isinstance(value, (list, tuple)):
        return [leaf for item in value for leaf in _trainable_leaves(item)]
    return []


def practice_tensor(name, yours, reference, cases, atol=1e-6):
    """practice() for functions that build Tensors.

    Each case runs on fresh copies of its Tensor arguments. The check compares
    the values your function returns AND the gradients that flow back to every
    argument with requires_grad=True, so a function that cuts the graph (for
    example by working on .data) is caught as well.
    """
    def checked(function):
        def run(*args):
            copies = [_fresh(arg) for arg in args]
            out = function(*copies)
            outs = out if isinstance(out, tuple) else (out,)
            if not all(isinstance(item, Tensor) for item in outs):
                raise TypeError("return Tensor results built from Tensor operations, so gradients can flow")
            total = Tensor(0.0)
            for index, item in enumerate(outs):
                weights = np.random.default_rng(index).normal(size=item.shape)
                total = total + (item * weights).sum()
            total.backward()
            grads = [leaf.grad if leaf.grad is not None else np.zeros_like(leaf.data)
                     for leaf in _trainable_leaves(copies)]
            return {"values": [item.data for item in outs], "gradients": grads}
        return run

    mine, ref = checked(yours), checked(reference)
    return yours if practice(name, mine, ref, cases, atol) is mine else reference


COLOURS = ("red", "green", "blue", "yellow")
SIZES = ("small", "large")
ORIENTATIONS = ("horizontal", "vertical")
ATTRIBUTES = {"colour": COLOURS, "size": SIZES, "orientation": ORIENTATIONS}
RGB = np.array([[0.85, 0.12, 0.10],   # red
                [0.15, 0.70, 0.20],   # green
                [0.20, 0.30, 0.90],   # blue
                [0.90, 0.80, 0.10]])  # yellow
BAR = {0: (2, 8), 1: (3, 12)}  # size index -> (thickness, length) in pixels
IMAGE_SIZE = 16

# Every (colour, size, orientation) combination, as attribute indices.
ALL_COMBOS = [(c, s, o) for c in range(4) for s in range(2) for o in range(2)]
# Four combinations never shown in training. Every single attribute value
# still appears in training; only these particular combinations are new.
HELD_OUT_COMBOS = [(0, 0, 1), (1, 1, 0), (2, 0, 0), (3, 1, 1)]
TRAIN_COMBOS = [combo for combo in ALL_COMBOS if combo not in HELD_OUT_COMBOS]


def bar_box(size, orientation):
    """(height, width) of the bar in pixels."""
    thickness, length = BAR[int(size)]
    return (thickness, length) if orientation == 0 else (length, thickness)


def sample_scenes(n, combos, seed):
    """Describe n images: attribute indices, the bar's top-left corner and a noise seed."""
    rng = np.random.default_rng(seed)
    chosen = np.array(combos)[rng.integers(0, len(combos), n)]
    scenes = {"colour": chosen[:, 0], "size": chosen[:, 1], "orientation": chosen[:, 2]}
    boxes = np.array([bar_box(s, o) for s, o in zip(scenes["size"], scenes["orientation"])])
    scenes["row"] = rng.integers(0, IMAGE_SIZE - boxes[:, 0] + 1)
    scenes["col"] = rng.integers(0, IMAGE_SIZE - boxes[:, 1] + 1)
    scenes["noise_seed"] = rng.integers(0, 2**31, n)
    return scenes


def render(scenes):
    """Draw every scene as a (16, 16, 3) RGB image with values in [0, 1]."""
    n = len(scenes["colour"])
    images = np.empty((n, IMAGE_SIZE, IMAGE_SIZE, 3))
    for i in range(n):
        rng = np.random.default_rng(int(scenes["noise_seed"][i]))
        image = 0.08 + 0.06 * rng.random((IMAGE_SIZE, IMAGE_SIZE, 3))  # dark, slightly noisy background
        shade = rng.uniform(0.85, 1.15)  # brightness varies from image to image
        tint = rng.uniform(-0.1, 0.1, 3)  # and so does the exact hue
        colour = RGB[scenes["colour"][i]] * shade + tint
        height, width = bar_box(scenes["size"][i], scenes["orientation"][i])
        r, c = scenes["row"][i], scenes["col"][i]
        image[r:r + height, c:c + width] = colour + 0.04 * rng.normal(size=(height, width, 3))
        images[i] = np.clip(image, 0.0, 1.0)
    return images


def describe(scenes):
    """The caption of every scene, for example 'a large red horizontal bar'."""
    return [f"a {SIZES[s]} {COLOURS[c]} {ORIENTATIONS[o]} bar"
            for c, s, o in zip(scenes["colour"], scenes["size"], scenes["orientation"])]


def combo_ids(scenes):
    """One number from 0 to 15 per scene: which of the 16 combinations it shows."""
    return scenes["colour"] * 4 + scenes["size"] * 2 + scenes["orientation"]


train_scenes = sample_scenes(800, TRAIN_COMBOS, seed=101)
seen_scenes = sample_scenes(300, TRAIN_COMBOS, seed=202)        # new images, combinations seen in training
held_out_scenes = sample_scenes(300, HELD_OUT_COMBOS, seed=303)  # new images, combinations never seen
train_images, seen_images, held_out_images = render(train_scenes), render(seen_scenes), render(held_out_scenes)


def channel_stats(images):
    """Mean and standard deviation of each colour channel, over every pixel of every image."""
    return images.mean(axis=(0, 1, 2)), images.std(axis=(0, 1, 2))


def normalise_images(images, mean, std):
    """(images - mean) / std for every channel. images: (n, H, W, 3); mean, std: (3,)."""
    return (images - mean) / std


def patchify(images, patch_size):
    """Cut images (n, H, W, C) into non-overlapping square patches: (n, N, P*P*C), N = (H/P)*(W/P).

    Patches are numbered row by row; inside a patch the values run row by row,
    then column by column, then channel by channel.
    """
    n, height, width, channels = images.shape
    p = patch_size
    grid = images.reshape(n, height // p, p, width // p, p, channels)
    grid = grid.transpose(0, 1, 3, 2, 4, 5)  # (n, rows, cols, p, p, C)
    return grid.reshape(n, (height // p) * (width // p), p * p * channels)


def unpatchify(patches, patch_size, height, width):
    """The inverse of patchify: (n, N, P*P*C) back to images (n, H, W, C)."""
    n, _, patch_dim = patches.shape
    p = patch_size
    channels = patch_dim // (p * p)
    grid = patches.reshape(n, height // p, width // p, p, p, channels)
    grid = grid.transpose(0, 1, 3, 2, 4, 5)  # (n, rows, p, cols, p, C)
    return grid.reshape(n, height, width, channels)


def prepare(images, mean, std, patch_size=4):
    """Normalise with training statistics, then cut into patches."""
    return patchify(normalise_images(images, mean, std), patch_size)


def init_image_encoder(seed, patch_dim=48, n_patches=16, width=32):
    """Weights of the tiny image encoder, all trainable Tensors."""
    rng = np.random.default_rng(seed)
    return {
        "W_patch": parameter(patch_dim, width, rng=rng),
        "b_patch": zeros_parameter(width),
        "pos": parameter(n_patches, width, rng=rng, scale=0.1),
        "W_mix": parameter(width, width, rng=rng),
        "b_mix": zeros_parameter(width),
    }


def embed_patches(patches, W, b, pos):
    """Linear patch embedding plus one learned position vector per patch.

    patches: Tensor (n, N, patch_dim); W: (patch_dim, D); b: (D,); pos: (N, D).
    Returns a Tensor (n, N, D).
    """
    return patches @ W + b + pos


def encode_images(patches, params):
    """The tiny encoder: embed the patches, apply a ReLU layer to each patch, then average.

    patches: Tensor (n, N, patch_dim); params: the dictionary from init_image_encoder.
    Returns (features, pooled): Tensors of shape (n, N, D) and (n, D).
    """
    tokens = embed_patches(patches, params["W_patch"], params["b_patch"], params["pos"])
    features = (tokens @ params["W_mix"] + params["b_mix"]).relu()
    return features, features.mean(axis=1)


class Adam:
    """The Adam optimiser for a list of Tensors, as in frameworks such as PyTorch."""

    def __init__(self, params, lr=0.01, betas=(0.9, 0.999), eps=1e-8):
        self.params, self.lr, self.betas, self.eps = list(params), lr, betas, eps
        self.m = [np.zeros_like(p.data) for p in self.params]  # running mean of gradients
        self.v = [np.zeros_like(p.data) for p in self.params]  # running mean of squared gradients
        self.t = 0

    def zero_grad(self):
        zero_grad(self.params)

    def step(self):
        self.t += 1
        beta1, beta2 = self.betas
        for p, m, v in zip(self.params, self.m, self.v):
            if p.grad is None:
                continue
            m[...] = beta1 * m + (1 - beta1) * p.grad
            v[...] = beta2 * v + (1 - beta2) * p.grad ** 2
            m_hat, v_hat = m / (1 - beta1 ** self.t), v / (1 - beta2 ** self.t)
            p.data -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


VOCAB = ["<pad>", "a", "bar"] + list(SIZES) + list(COLOURS) + list(ORIENTATIONS)
TOKEN_ID = {word: index for index, word in enumerate(VOCAB)}
CAPTION_LENGTH = 6  # the longest caption has 5 words; the rest is padding


def tokenise(captions, length=CAPTION_LENGTH):
    """Word ids, padded with 0 (<pad>) to `length`: an int array (n, length)."""
    ids = np.zeros((len(captions), length), dtype=int)
    for i, caption in enumerate(captions):
        words = caption.split()
        ids[i, :len(words)] = [TOKEN_ID[word] for word in words]
    return ids


def encode_text(token_ids, E, W):
    """Bag-of-tokens text encoder: average the embeddings of the real (non-pad) tokens, then map.

    token_ids: int array (n, L) with 0 for <pad>; E: Tensor (V, D); W: Tensor (D, D_out).
    Returns a Tensor (n, D_out).
    """
    mask = (token_ids != 0).astype(float)[..., None]  # (n, L, 1): 1.0 for real tokens
    mean = (E[token_ids] * mask).sum(axis=1) / mask.sum(axis=1)  # (n, D)
    return mean @ W


def l2_normalise(x, eps=1e-8):
    """Scale every row of x to unit length, so dot products become cosine similarities."""
    return x / ((x * x).sum(axis=-1, keepdims=True) + eps) ** 0.5


def clip_loss(logits):
    """Symmetric InfoNCE with the matching pair on the diagonal. logits: Tensor (n, n) -> scalar Tensor."""
    targets = np.arange(logits.shape[0])
    return (cross_entropy(logits, targets) + cross_entropy(logits.T, targets)) * 0.5


def positive_mask(caption_ids):
    """mask[i, j] = 1.0 when items i and j have the same caption, else 0.0. caption_ids: (n,) -> (n, n)."""
    ids = np.asarray(caption_ids)
    return (ids[:, None] == ids[None, :]).astype(float)


def multi_positive_loss(logits, positives):
    """Symmetric contrastive loss in which every 1.0 in `positives` counts as a correct match.

    logits: Tensor (n, n), images in rows and captions in columns; positives: (n, n) array.
    """
    image_to_text = -(logits.softmax(axis=1) * positives).sum(axis=1).log().mean()
    text_to_image = -(logits.softmax(axis=0) * positives).sum(axis=0).log().mean()
    return (image_to_text + text_to_image) * 0.5


def recall_at_k(scores, relevant, k):
    """Fraction of queries (rows) with at least one relevant candidate among their k best scores.

    scores: (n_queries, n_candidates); relevant: same shape, True where a candidate is a valid match.
    """
    top_k = np.argsort(-scores, axis=1, kind="stable")[:, :k]
    return float(np.take_along_axis(relevant, top_k, axis=1).any(axis=1).mean())


ALL_CAPTIONS = [f"a {SIZES[s]} {COLOURS[c]} {ORIENTATIONS[o]} bar" for c, s, o in ALL_COMBOS]


def init_alignment_model(seed, embed_dim=16):
    """An image encoder with a projection head, and a bag-of-tokens text encoder."""
    rng = np.random.default_rng(seed + 1)
    return {
        "image": init_image_encoder(seed),
        "W_image": parameter(32, embed_dim, rng=rng),
        "E": parameter(len(VOCAB), embed_dim, rng=rng, scale=0.5),
        "W_text": parameter(embed_dim, embed_dim, rng=rng),
    }


def model_parameters(model):
    """Every Tensor in a (nested) dictionary of parameters."""
    found = []
    for value in model.values():
        if isinstance(value, dict):
            found += model_parameters(value)
        elif isinstance(value, Tensor):
            found.append(value)
    return found


def embed_images(model, patches):
    _, pooled = encode_images(Tensor(patches), model["image"])
    return l2_normalise(pooled @ model["W_image"])


def embed_captions(model, token_ids):
    return l2_normalise(encode_text(token_ids, model["E"], model["W_text"]))


def train_alignment(model, patches, token_ids, caption_ids, steps=200, batch_size=64, lr=0.01,
                    temperature=0.1, multi_positive=True, seed=0):
    """Train both encoders with the contrastive loss; returns the loss of every step."""
    optimiser = Adam(model_parameters(model), lr)
    order = np.random.default_rng(seed)
    losses = []
    for step in range(steps):
        batch = order.choice(len(token_ids), batch_size, replace=False)
        logits = embed_images(model, patches[batch]) @ embed_captions(model, token_ids[batch]).T / temperature
        if multi_positive:
            loss = multi_positive_loss(logits, positive_mask(caption_ids[batch]))
        else:
            loss = clip_loss(logits)
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        losses.append(float(loss.data))
    return losses


def retrieval_report(model, patches, scenes):
    """Image-to-text retrieval over the 16 possible captions: R@1, R@3 and per-attribute top-1 accuracy."""
    scores = embed_images(model, patches).data @ embed_captions(model, tokenise(ALL_CAPTIONS)).data.T
    relevant = combo_ids(scenes)[:, None] == np.arange(len(ALL_CAPTIONS))[None, :]
    best = np.array(ALL_COMBOS)[scores.argmax(axis=1)]
    report = {"R@1": recall_at_k(scores, relevant, 1), "R@3": recall_at_k(scores, relevant, 3)}
    for index, attribute in enumerate(ATTRIBUTES):
        report[attribute] = float((best[:, index] == scenes[attribute]).mean())
    return report

### Start from the aligned image encoder

The next cell repeats notebook 2's training run with the same seeds, so you get the same aligned encoder back, and prints its held-out retrieval as a reminder of what it learned.

In [ ]:
mean, std = channel_stats(train_images)
train_patches, seen_patches, held_out_patches = (prepare(images, mean, std)
                                                 for images in (train_images, seen_images, held_out_images))
alignment = init_alignment_model(seed=0)
start = time.perf_counter()
train_alignment(alignment, train_patches, tokenise(describe(train_scenes)), combo_ids(train_scenes), seed=2)
print(f"aligned in {time.perf_counter() - start:.2f} s; held-out retrieval:",
      {key: round(value, 3) for key, value in retrieval_report(alignment, held_out_patches, held_out_scenes).items()})

## Part 1 · Questions, answers and the sequence layout

Every training image gets one of three questions, each with a one-word answer:

| question | tokens | possible answers |
|---|---|---|
| what colour is the bar ? | 6 | red, green, blue, yellow |
| how big is it ? | 5 | small, large |
| is the bar horizontal or vertical ? | 7 | horizontal, vertical |

The language model has its own vocabulary of 21 words. A text row holds the question, the answer and an `<end>` token, padded with `<pad>` to `TEXT_LENGTH = 9`. The given cell builds these rows and decodes a few. Note that the orientation question names both possible answers, so the words alone give away nothing about which one is right.

In [ ]:
LM_VOCAB = (["<pad>", "<end>", "what", "colour", "is", "the", "bar", "?", "how", "big", "it", "or", "none"]
            + list(COLOURS) + list(SIZES) + list(ORIENTATIONS))
LM_ID = {word: index for index, word in enumerate(LM_VOCAB)}
QUESTION_TYPES = ("colour", "size", "orientation")
QUESTIONS = {"colour": "what colour is the bar ?", "size": "how big is it ?",
             "orientation": "is the bar horizontal or vertical ?"}
TEXT_LENGTH = 9  # the longest question (7 tokens) + answer + <end>


def make_vqa_batch(scenes, seed, no_bar=None):
    """One question per scene. Returns (text, question_lengths, question_types, answers):

    text: int (n, TEXT_LENGTH) = question + answer + <end> + <pad>s; question_lengths: (n,);
    question_types: (n,) indices into QUESTION_TYPES; answers: (n,) ids of the correct answer word.
    Scenes flagged in `no_bar` get the answer "none".
    """
    rng = np.random.default_rng(seed)
    n = len(scenes["colour"])
    types = rng.integers(0, len(QUESTION_TYPES), n)
    text = np.zeros((n, TEXT_LENGTH), dtype=int)
    lengths, answers = np.zeros(n, dtype=int), np.zeros(n, dtype=int)
    for i in range(n):
        attribute = QUESTION_TYPES[types[i]]
        question = QUESTIONS[attribute].split()
        if no_bar is not None and no_bar[i]:
            answer = "none"
        else:
            answer = ATTRIBUTES[attribute][scenes[attribute][i]]
        ids = [LM_ID[word] for word in question] + [LM_ID[answer], LM_ID["<end>"]]
        text[i, :len(ids)] = ids
        lengths[i], answers[i] = len(question), LM_ID[answer]
    return text, lengths, types, answers


def decode(ids):
    return " ".join(LM_VOCAB[i] for i in ids if i != 0)


train_text, train_lengths, train_types, train_answers = make_vqa_batch(train_scenes, seed=7)
seen_vqa = make_vqa_batch(seen_scenes, seed=8)
held_out_vqa = make_vqa_batch(held_out_scenes, seed=9)
for i in range(4):
    print(f"{describe(train_scenes)[i]:30s} -> {decode(train_text[i])}")
print("text:", train_text.shape, " question lengths of the first rows:", train_lengths[:4])

The model's input is a sequence of $N$ visual tokens followed by the text tokens. Like any decoder-only language model it predicts, at every position $t$, the token at position $t + 1$. With one visual token ($N = 1$) and the colour question, the sequence lines up like this:

| position $t$ | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|---|
| input | *image* | what | colour | is | the | bar | ? | red | `<end>` |
| target (token $t+1$) | what | colour | is | the | bar | ? | red | `<end>` | `<pad>` |
| counted in the loss | no | no | no | no | no | no | **yes** | **yes** | no |

The last of the nine text tokens is never an input, so the input has $N + 8$ positions. The question tokens stay in the input as context but are not trained: the model should learn to *answer*, not to predict the question. For a question of length $q$, the answer is predicted at position $N + q - 1$ and `<end>` at $N + q$.

## Part 2 · The projector

The aligned image embedding from notebook 2 has 16 numbers; our language model works with vectors of width 24. A **projector** maps one to the other with a learned matrix and bias:

$$\mathbf{v}_k = \mathbf{f}_k W + \mathbf{b}, \qquad W \in \mathbb{R}^{d_\text{vision} \times d_\text{model}}.$$

It is applied to each visual token $k$ separately: features of shape (n, N, d_vision) become (n, N, d_model). LLaVA, the paper that popularised this design, used exactly such a linear layer on the patch features of a CLIP encoder; its successor LLaVA-1.5 used a two-layer MLP instead. Other designs compress many visual features into a few learned query tokens (BLIP-2's Q-Former) or let the language model attend to them through cross-attention layers (Flamingo). Matching the widths is the easy part: training must still make the projected vectors *mean* something to the language model.

In this notebook the connector receives **one** visual token, the aligned 16-number image embedding, as in "prefix" captioning models built on CLIP embeddings. Experiment 1 tries the LLaVA-style alternative, one token per patch.

In [ ]:
features_demo = Tensor(np.ones((2, 1, 16)))  # 2 images, 1 visual token, 16 numbers each
W_demo = Tensor(np.full((16, 24), 0.1))
print("(2, 1, 16) @ (16, 24) ->", (features_demo @ W_demo).shape)

### ✍️ Your turn: `project_visual_tokens`

Write `project_visual_tokens(features, W, b)`.

- `features`: a `Tensor` of shape (n, N, d_vision);
- `W`: a `Tensor` of shape (d_vision, d_model); `b`: a `Tensor` of shape (d_model,);
- return the `Tensor` $\mathbf{f}W + \mathbf{b}$ of shape (n, N, d_model).

*Hint:* one line of `Tensor` operations; the bias broadcasts over the image and token axes.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def project_visual_tokens_reference(features, W, b):
    """Map vision features into the language model's width.

    features: Tensor (n, N, d_vision); W: (d_vision, d_model); b: (d_model,). Returns (n, N, d_model).
    """
    return features @ W + b

In [ ]:
def project_visual_tokens(features, W, b):
    """Map vision features into the language model's width.

    features: Tensor (n, N, d_vision); W: (d_vision, d_model); b: (d_model,). Returns (n, N, d_model).
    """
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
case_rng = np.random.default_rng(31)
project_visual_tokens = practice_tensor("project_visual_tokens", project_visual_tokens, project_visual_tokens_reference, cases=[
    (Tensor(case_rng.normal(size=(2, 1, 16))), parameter(16, 24, rng=case_rng), parameter(24, rng=case_rng, scale=0.1)),
    (Tensor(case_rng.normal(size=(3, 16, 32))), parameter(32, 24, rng=case_rng), parameter(24, rng=case_rng, scale=0.1)),
    (Tensor(case_rng.normal(size=(1, 1, 2))), parameter(2, 3, rng=case_rng), parameter(3, rng=case_rng, scale=0.1)),  # edge case
])

**Expected behaviour.** ✅ on three cases: one visual token per image, sixteen patch tokens per image, and a tiny edge case. The same projector works for any number of visual tokens, because it treats each token independently.

## Part 3 · Build the input sequence

The engine has no operation that joins two `Tensor`s along an axis, so the next cell adds one, `concat`, built on top of `Tensor` exactly like the engine's own operations: the forward pass calls `np.concatenate`, and the backward pass splits the incoming gradient back into one piece per input. A new operation must be gradient-checked before anything trains with it, so the cell compares its gradients with `numerical_grad`.

In [ ]:
def concat(tensors, axis):
    """Join Tensors along an existing axis; the gradient is split back into the pieces."""
    tensors = [Tensor.lift(t) for t in tensors]
    sizes = [t.shape[axis] for t in tensors]
    data = np.concatenate([t.data for t in tensors], axis=axis)

    def backward(g):
        return tuple(np.split(g, np.cumsum(sizes)[:-1], axis=axis))

    return tensors[0]._result(data, tuple(tensors), backward, "concat")


check_rng = np.random.default_rng(32)
left, right = Tensor(check_rng.normal(size=(2, 1, 3)), requires_grad=True), Tensor(check_rng.normal(size=(2, 4, 3)), requires_grad=True)
weights = check_rng.normal(size=(2, 5, 3))
objective = lambda: (concat([left, right], axis=1) * weights).sum()
objective().backward()
errors = [relative_error(t.grad, numerical_grad(objective, t)) for t in (left, right)]
print("concat gradient check, relative errors:", [f"{e:.1e}" for e in errors])
assert max(errors) < 1e-6, "concat's backward pass is wrong"

The language model's input is the projected visual tokens followed by the embedded text tokens, plus one learned position vector per place in the sequence:

$$X = [\,\mathbf{v}_1, \dots, \mathbf{v}_N, E_{w_1}, \dots, E_{w_T}\,] + P, \qquad X \in \mathbb{R}^{n \times (N+T) \times d}.$$

Here the image comes first and the question after it, as in LLaVA. Real processors insert an `<image>` placeholder into the prompt and expand it into the right number of visual tokens; a mismatch between placeholders and supplied images is a classic silent bug.

### ✍️ Your turn: `build_input_sequence`

Write `build_input_sequence(visual_tokens, text_ids, E, pos)`.

- `visual_tokens`: a `Tensor` of shape (n, N, d), the projector's output;
- `text_ids`: an integer array of shape (n, T); `E`: the language model's embedding table, a `Tensor` of shape (V, d);
- `pos`: a `Tensor` of shape (N + T, d);
- return the `Tensor` of shape (n, N + T, d): the visual tokens followed by `E[text_ids]` along axis 1, plus `pos`.

*Hint:* use `concat([...], axis=1)` from the cell above, then add `pos`, which broadcasts over the batch axis.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def build_input_sequence_reference(visual_tokens, text_ids, E, pos):
    """The language model's input: visual tokens first, then the embedded text, plus positions.

    visual_tokens: Tensor (n, N, d); text_ids: int (n, T); E: (V, d); pos: (N + T, d).
    Returns a Tensor (n, N + T, d).
    """
    return concat([visual_tokens, E[text_ids]], axis=1) + pos

In [ ]:
def build_input_sequence(visual_tokens, text_ids, E, pos):
    """The language model's input: visual tokens first, then the embedded text, plus positions.

    visual_tokens: Tensor (n, N, d); text_ids: int (n, T); E: (V, d); pos: (N + T, d).
    Returns a Tensor (n, N + T, d).
    """
    # ✍️ Your code here (one or two lines)
    raise NotImplementedError

In [ ]:
case_rng = np.random.default_rng(33)
E_case = parameter(len(LM_VOCAB), 4, rng=case_rng, scale=0.3)
build_input_sequence = practice_tensor("build_input_sequence", build_input_sequence, build_input_sequence_reference, cases=[
    (Tensor(case_rng.normal(size=(2, 1, 4)), requires_grad=True), train_text[:2, :-1], E_case, parameter(9, 4, rng=case_rng, scale=0.1)),
    (Tensor(case_rng.normal(size=(2, 3, 4)), requires_grad=True), train_text[2:4, :3], E_case, parameter(6, 4, rng=case_rng, scale=0.1)),
    (Tensor(case_rng.normal(size=(1, 1, 4)), requires_grad=True), train_text[:1, :1], E_case, parameter(2, 4, rng=case_rng, scale=0.1)),  # edge case
])

**Expected behaviour.** ✅ on three cases. The gradient check matters here: the gradient must reach the visual tokens, because that is the only way the projector, and an unfrozen vision encoder, can learn anything.

## Part 4 · Train on the answer only

The engine's `cross_entropy(logits, targets, mask)` averages the loss only over positions where `mask` is 1. With $N$ visual tokens and a question of $q$ tokens, the answer is predicted at position $N + q - 1$ and `<end>` at $N + q$:

$$m_{t} = \begin{cases} 1 & t = N + q - 1 \text{ or } t = N + q,\\ 0 & \text{otherwise.}\end{cases}$$

Because the three questions have different lengths, the supervised positions differ from row to row. Frameworks implement the same idea by setting the targets of non-answer positions to an "ignore" index; whichever way you do it, check the supervised positions *after* the sequence has been assembled, because a shift by one silently trains the model to predict the wrong token.

### ✍️ Your turn: `answer_loss_mask`

Write `answer_loss_mask(question_lengths, n_visual, seq_len)`.

- `question_lengths`: an integer array of shape (n,); `n_visual`: the number of visual tokens $N$; `seq_len`: the input length $L$;
- return a float array of shape (n, L) with 1.0 at positions $N + q_i - 1$ and $N + q_i$ of row $i$, and 0.0 everywhere else.

*Hint:* compare `np.arange(seq_len)[None, :]` with a column of answer positions of shape (n, 1).

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def answer_loss_mask_reference(question_lengths, n_visual, seq_len):
    """mask[i, t] = 1.0 where position t predicts row i's answer or its <end> token, else 0.0.

    Position t predicts the token at t + 1. The answer follows the question, so it is
    predicted from the last question token, at n_visual + question_length - 1.
    """
    positions = np.arange(seq_len)[None, :]
    last_question = n_visual + np.asarray(question_lengths)[:, None] - 1
    return ((positions == last_question) | (positions == last_question + 1)).astype(float)

In [ ]:
def answer_loss_mask(question_lengths, n_visual, seq_len):
    """mask[i, t] = 1.0 where position t predicts row i's answer or its <end> token, else 0.0.

    Position t predicts the token at t + 1. The answer follows the question, so it is
    predicted from the last question token, at n_visual + question_length - 1.
    """
    # ✍️ Your code here (two lines)
    raise NotImplementedError

In [ ]:
answer_loss_mask = practice("answer_loss_mask", answer_loss_mask, answer_loss_mask_reference, cases=[
    (np.array([6, 5, 7]), 1, 9),
    (np.array([6, 5]), 16, 24),
    (np.array([1]), 0, 2),  # edge case: no visual tokens and a one-token prompt
])
print(answer_loss_mask(np.array([6, 5, 7]), 1, 9))

**Expected behaviour.** ✅, and the printed mask has two ones per row, starting at position 6 for the colour question, 5 for the size question and 7 for the orientation question. The next cell checks the invariant on real training rows: exactly two supervised positions each, and the targets there are the answer word and `<end>`.

In [ ]:
n_visual = 1
seq_len = n_visual + TEXT_LENGTH - 1
targets = np.concatenate([np.zeros((len(train_text), n_visual), dtype=int), train_text], axis=1)[:, 1:]
mask = answer_loss_mask(train_lengths, n_visual, seq_len)
assert (mask.sum(axis=1) == 2).all(), "every row should supervise exactly two positions"
supervised = targets[mask == 1].reshape(-1, 2)
assert (supervised[:, 0] == train_answers).all() and (supervised[:, 1] == LM_ID["<end>"]).all()
print("row 0 targets:", [LM_VOCAB[i] for i in targets[0]])
print("supervised:   ", [LM_VOCAB[i] for i in supervised[0]])

## Part 5 · The small language model

The language model is given: a decoder-only transformer with the same parts as the causal transformer in the Transformer Models course. It has a token embedding table, learned positions, two blocks of pre-norm causal self-attention with two heads and a ReLU MLP, a final layer norm and an output layer over the 21-word vocabulary. Unlike a real vision-language model, it starts from random weights: real systems start from a language model pretrained on large amounts of text, which is why their first training stage can update the projector alone.

The causal mask is what makes answer-only training honest. Without it, the position that predicts the answer could look ahead at the answer token itself. The cell after the code checks that invariant directly: changing later positions must leave every earlier output unchanged.

In [ ]:
def init_language_model(seed, n_visual, d_model=24, hidden=48, n_layers=2):
    """A small decoder-only transformer: embeddings, positions, two causal blocks and an output layer."""
    rng = np.random.default_rng(seed)
    lm = {"E": parameter(len(LM_VOCAB), d_model, rng=rng, scale=0.3),
          "pos": parameter(n_visual + TEXT_LENGTH - 1, d_model, rng=rng, scale=0.1)}
    for layer in range(n_layers):
        lm[f"block{layer}"] = {
            "ln1_g": Tensor(np.ones(d_model), requires_grad=True), "ln1_b": zeros_parameter(d_model),
            "Wq": parameter(d_model, d_model, rng=rng), "Wk": parameter(d_model, d_model, rng=rng),
            "Wv": parameter(d_model, d_model, rng=rng), "Wo": parameter(d_model, d_model, rng=rng, scale=0.1),
            "ln2_g": Tensor(np.ones(d_model), requires_grad=True), "ln2_b": zeros_parameter(d_model),
            "W1": parameter(d_model, hidden, rng=rng), "b1": zeros_parameter(hidden),
            "W2": parameter(hidden, d_model, rng=rng, scale=0.1),
        }
    lm["lnf_g"], lm["lnf_b"] = Tensor(np.ones(d_model), requires_grad=True), zeros_parameter(d_model)
    lm["W_out"] = parameter(d_model, len(LM_VOCAB), rng=rng, scale=0.1)
    return lm


def language_model_forward(lm, x, heads=2):
    """Next-token logits (n, L, V) for an input sequence x: Tensor (n, L, d)."""
    n, length, d = x.shape
    head_dim = d // heads
    causal = np.triu(np.full((length, length), -1e9), 1)  # no position may look ahead

    def split_heads(t):
        return t.reshape(n, length, heads, head_dim).transpose(0, 2, 1, 3)

    for name in sorted(key for key in lm if key.startswith("block")):
        block = lm[name]
        h = layer_norm(x, block["ln1_g"], block["ln1_b"])
        q, k, v = split_heads(h @ block["Wq"]), split_heads(h @ block["Wk"]), split_heads(h @ block["Wv"])
        weights = (q @ k.transpose(0, 1, 3, 2) / np.sqrt(head_dim) + causal).softmax(axis=-1)
        x = x + (weights @ v).transpose(0, 2, 1, 3).reshape(n, length, d) @ block["Wo"]
        h = layer_norm(x, block["ln2_g"], block["ln2_b"])
        x = x + (h @ block["W1"] + block["b1"]).relu() @ block["W2"]
    return layer_norm(x, lm["lnf_g"], lm["lnf_b"]) @ lm["W_out"]

In [ ]:
causal_rng = np.random.default_rng(34)
lm_check = init_language_model(seed=34, n_visual=1)
x = causal_rng.normal(size=(2, 9, 24))
x_changed = x.copy()
x_changed[:, 6:] += causal_rng.normal(size=(2, 3, 24))  # change positions 6, 7 and 8 only
before = language_model_forward(lm_check, Tensor(x)).data
after = language_model_forward(lm_check, Tensor(x_changed)).data
assert np.allclose(before[:, :6], after[:, :6]), "positions 0-5 must not see later positions"
assert not np.allclose(before[:, 6:], after[:, 6:])
print("causal mask holds: positions 0-5 are unchanged; parameters in the language model:",
      sum(p.data.size for p in model_parameters(lm_check)))

### ✍️ Your turn: `read_answers`

At test time the model sees the image and the question, and its answer is the most likely token at the position that predicts the answer. Write `read_answers(logits, question_lengths, n_visual)`.

- `logits`: a NumPy array of shape (n, L, V), the language model's output as plain numbers;
- `question_lengths`: an integer array of shape (n,); `n_visual`: $N$;
- return an integer array of shape (n,): for row $i$, the index of the largest logit at position $N + q_i - 1$.

*Hint:* index with two arrays at once, `logits[rows, positions]`, to pick one position per row, then `argmax` over the vocabulary axis.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def read_answers_reference(logits, question_lengths, n_visual):
    """The model's answer: the most likely token at the position that predicts the answer.

    logits: array (n, L, V); question_lengths: (n,). Returns int ids (n,).
    """
    rows = np.arange(len(question_lengths))
    return logits[rows, n_visual + np.asarray(question_lengths) - 1].argmax(axis=-1)

In [ ]:
def read_answers(logits, question_lengths, n_visual):
    """The model's answer: the most likely token at the position that predicts the answer.

    logits: array (n, L, V); question_lengths: (n,). Returns int ids (n,).
    """
    # ✍️ Your code here (two lines)
    raise NotImplementedError

In [ ]:
case_rng = np.random.default_rng(35)
read_answers = practice("read_answers", read_answers, read_answers_reference, cases=[
    (case_rng.normal(size=(3, 9, 21)), np.array([6, 5, 7]), 1),
    (case_rng.normal(size=(2, 24, 21)), np.array([5, 7]), 16),
    (case_rng.normal(size=(1, 2, 4)), np.array([1]), 0),  # edge case: the answer is predicted at position 0
])

**Expected behaviour.** ✅ on three cases. Reading one position too late returns the model's guess for `<end>`, so its "answers" would almost all be `<end>`; that is the first thing to check when a VQA model scores zero.

## Part 6 · Train: vision encoder frozen or unfrozen

The given code assembles a full model: a *copy* of the aligned vision encoder, a fresh projector and a fresh language model. `visual_inputs` chooses what the connector receives: the aligned image embedding as one token, or the encoder's patch features as sixteen tokens. `train_vqa` runs 300 steps of answer-only training on batches of 32 questions, with two Adam optimisers: step size 0.01 for the new projector and language model, and a smaller `vision_lr` for the pretrained encoder. The encoder's weights are only updated when they require gradients.

Which modules should train? LLaVA trained in two stages: first the projector alone, with the vision encoder and the language model frozen, then the projector and the language model together, still with the vision encoder frozen. Our language model starts untrained, so it must learn in every run, and the comparison here is about the vision encoder: **frozen** keeps the aligned features exactly as notebook 2 left them; **unfrozen** lets the answer loss adjust them too.

In [ ]:
def visual_inputs(model, patches):
    """What the connector receives: the aligned image embedding from notebook 2 as one token,
    or the encoder's patch features as one token per patch."""
    features, pooled = encode_images(Tensor(patches), model["vision"]["image"])
    if model["visual_tokens"] == "embedding":
        embedding = l2_normalise(pooled @ model["vision"]["W_image"])  # (n, 16)
        return embedding.reshape(embedding.shape[0], 1, embedding.shape[1])
    return features  # (n, 16 patches, 32)


def vqa_logits(model, patches, text_in):
    visual = project_visual_tokens(visual_inputs(model, patches), model["projector"]["W"], model["projector"]["b"])
    x = build_input_sequence(visual, text_in, model["language_model"]["E"], model["language_model"]["pos"])
    return language_model_forward(model["language_model"], x)


def new_vqa_model(alignment_model, seed, visual_tokens="embedding", freeze_vision=True, d_model=24):
    """A fresh projector and language model on top of a copy of the aligned image encoder."""
    def copy(t):
        return Tensor(t.data.copy(), requires_grad=not freeze_vision)

    n_visual, d_vision = (1, 16) if visual_tokens == "embedding" else (16, 32)
    rng = np.random.default_rng(seed)
    return {
        "visual_tokens": visual_tokens, "n_visual": n_visual,
        "vision": {"image": {key: copy(value) for key, value in alignment_model["image"].items()},
                   "W_image": copy(alignment_model["W_image"])},
        "projector": {"W": parameter(d_vision, d_model, rng=rng), "b": zeros_parameter(d_model)},
        "language_model": init_language_model(seed + 1, n_visual, d_model),
    }


def train_vqa(model, patches, text, question_lengths, steps=300, batch_size=32, lr=0.01, vision_lr=0.001, seed=0):
    """Answer-only training. The vision encoder is updated only if its Tensors require gradients."""
    new_parts = model_parameters(model["projector"]) + model_parameters(model["language_model"])
    optimisers = [Adam(new_parts, lr)]
    vision = [p for p in model_parameters(model["vision"]) if p.requires_grad]
    if vision:
        optimisers.append(Adam(vision, vision_lr))  # pretrained weights get a smaller step
    order = np.random.default_rng(seed)
    n_visual = model["n_visual"]
    seq_len = n_visual + TEXT_LENGTH - 1
    losses = []
    for step in range(steps):
        batch = order.choice(len(text), batch_size, replace=False)
        logits = vqa_logits(model, patches[batch], text[batch, :-1])
        targets = np.concatenate([np.zeros((batch_size, n_visual), dtype=int), text[batch]], axis=1)[:, 1:]
        loss = cross_entropy(logits, targets, answer_loss_mask(question_lengths[batch], n_visual, seq_len))
        for optimiser in optimisers:
            optimiser.zero_grad()
        loss.backward()
        for optimiser in optimisers:
            optimiser.step()
        losses.append(float(loss.data))
    return losses


def answer_questions(model, patches, text, question_lengths):
    """Predicted answer ids and the probability the model gave its answer."""
    logits = vqa_logits(model, patches, text[:, :-1]).data
    at = logits[np.arange(len(text)), model["n_visual"] + np.asarray(question_lengths) - 1]
    probs = np.exp(at - at.max(axis=-1, keepdims=True))
    probs /= probs.sum(axis=-1, keepdims=True)
    return read_answers(logits, question_lengths, model["n_visual"]), probs.max(axis=-1)


def vqa_report(model, patches, text, question_lengths, types, answers):
    """Accuracy for each question type and overall."""
    predicted, _ = answer_questions(model, patches, text, question_lengths)
    report = {name: float((predicted[types == t] == answers[types == t]).mean()) for t, name in enumerate(QUESTION_TYPES)}
    report["all"] = float((predicted == answers).mean())
    return report

### 🧪 Try it: the encoder's step size

The next cell trains both models from the same seed. After you have run it once, set `VISION_LR = 0.01`, run it and the results cell again, and compare the held-out row of the unfrozen model.

In [ ]:
VISION_LR = 0.001  # the unfrozen encoder's step size; try 0.01, then run this cell and the next one again

runs = {}
for name, freeze in (("frozen", True), ("unfrozen", False)):
    vqa = new_vqa_model(alignment, seed=10, freeze_vision=freeze)
    vision_before = [p.data.copy() for p in model_parameters(vqa["vision"])]
    start = time.perf_counter()
    vqa_losses = train_vqa(vqa, train_patches, train_text, train_lengths, vision_lr=VISION_LR, seed=3)
    runs[name] = {
        "model": vqa, "losses": vqa_losses, "seconds": time.perf_counter() - start,
        "vision_unchanged": all(np.array_equal(old, p.data) for old, p in zip(vision_before, model_parameters(vqa["vision"]))),
        "trainable": sum(p.data.size for p in model_parameters(vqa) if p.requires_grad),
    }
    print(f"{name:8s}: {runs[name]['trainable']:6,d} trainable parameters, {runs[name]['seconds']:.2f} s, "
          f"loss {np.mean(vqa_losses[:10]):.3f} -> {np.mean(vqa_losses[-10:]):.3f}")

In [ ]:
print(f"{'accuracy':28s}" + "".join(f"{key:>13s}" for key in (*QUESTION_TYPES, "all")))
for name, run in runs.items():
    for split, patches, questions in (("seen", seen_patches, seen_vqa), ("held-out", held_out_patches, held_out_vqa)):
        report = vqa_report(run["model"], patches, *questions)
        run[split] = report
        print(f"{name + ', ' + split:28s}" + "".join(f"{report[key]:13.3f}" for key in (*QUESTION_TYPES, "all")))

plt.figure(figsize=(5, 2.6))
for name, run in runs.items():
    plt.plot(run["losses"], label=name)
plt.yscale("log")
plt.xlabel("step")
plt.ylabel("answer loss")
plt.legend()
plt.show()

**Expected behaviour.** Each run takes under a second. The frozen model trains about 11,000 parameters and the unfrozen one about 15,000, because the encoder's 3,648 weights join in. Both answer almost every question about the seen combinations correctly. On the held-out combinations our run gave about 0.86 for the frozen model and 0.99 for the unfrozen one; most of the frozen model's misses were colour questions. Do not read a general rule into one run: across eight seeds the held-out accuracy ranged from 0.86 to 0.98 frozen and from 0.79 to 0.99 unfrozen, and neither setting won every time.

With `VISION_LR = 0.01` the pretrained encoder moves ten times further on each step. Across eight seeds, held-out accuracy was then lower than with 0.001 in seven, and in three of them held-out orientation collapsed below 0.4, while seen accuracy usually suffered less. The encoder adapted to the training combinations and lost part of what transferred to new ones. That is why pretrained parts usually get a smaller step size, or stay frozen.

## Put it together: are the answers about the image?

The final cell checks the two runs. First, the bookkeeping: the frozen encoder must be bit-for-bit unchanged and must never have received a gradient, while the unfrozen one must have changed. Second, accuracy floors on seen and held-out combinations, set below every seed we tried. Third, an **image-swap control**: shuffle the held-out images so that every question is asked about a *different* image, and keep the questions fixed. A model that reads the image should drop to about chance on colour (one in four), because the colour of a random other bar is unrelated to the right answer; a model that answered from the question alone would not be affected at all.

In [ ]:
shuffle = np.random.default_rng(1).permutation(len(held_out_patches))
for name, run in runs.items():
    run["swapped"] = vqa_report(run["model"], held_out_patches[shuffle], *held_out_vqa)
    print(f"{name:8s} held-out colour accuracy: {run['held-out']['colour']:.3f} with the right images, "
          f"{run['swapped']['colour']:.3f} with shuffled images")

frozen_run, unfrozen_run = runs["frozen"], runs["unfrozen"]
assert frozen_run["vision_unchanged"], "a frozen encoder must not change"
assert all(p.grad is None for p in model_parameters(frozen_run["model"]["vision"])), "a frozen encoder gets no gradients"
assert not unfrozen_run["vision_unchanged"], "the unfrozen encoder should have been updated"
for run in runs.values():
    assert run["swapped"]["colour"] <= 0.45, "with the wrong image, colour answers should fall to about chance"
if VISION_LR == 0.001:  # the accuracy floors assume the default step size
    for run in runs.values():
        assert np.mean(run["losses"][-10:]) < 0.25 * np.mean(run["losses"][:10]), "the answer loss should fall"
        assert run["seen"]["all"] >= 0.9, "seen combinations should be answered almost perfectly"
    assert frozen_run["held-out"]["all"] >= 0.7 and unfrozen_run["held-out"]["all"] >= 0.65, "held-out floors"
    print("All checks passed.")
else:
    print("The bookkeeping and image-swap checks passed; the accuracy floors assume VISION_LR = 0.001.")

**Expected behaviour.** With the right images, the frozen model answered about 0.73 of the held-out colour questions correctly in our run and the unfrozen model all of them; with shuffled images both fall to about 0.23, close to the chance level of 0.25. The answers depend on the image, which is the minimum requirement for calling them visual. It is not yet proof that they depend on the *right part* of the image: notebook 4 changes one attribute at a time to test that.

Keep the scope in mind. This model answers three fixed question templates about one bar in a 16 × 16 drawing, with a language model trained from scratch on 800 questions. It has no knowledge of language beyond those templates, cannot read text in images and has never seen a photograph.

## 🧪 Experiment 1: one embedding token or sixteen patch tokens?

LLaVA-style connectors pass the language model one token per patch, which keeps spatial detail. The cell trains a third model whose connector receives the encoder's 16 patch features (width 32) instead of the single aligned embedding, with the vision encoder frozen, and compares the cost and the accuracy.

In [ ]:
patch_model = new_vqa_model(alignment, seed=10, visual_tokens="patches", freeze_vision=True)
start = time.perf_counter()
patch_losses = train_vqa(patch_model, train_patches, train_text, train_lengths, seed=3)
patch_seconds = time.perf_counter() - start
patch_seen = vqa_report(patch_model, seen_patches, *seen_vqa)
patch_held_out = vqa_report(patch_model, held_out_patches, *held_out_vqa)
print(f"{'connector input':34s} {'tokens':>6s} {'seconds':>8s} {'held-out colour':>16s} {'size':>6s} {'orientation':>12s}")
for label, tokens, seconds, report in (("aligned embedding (frozen)", 1, frozen_run["seconds"], frozen_run["held-out"]),
                                      ("16 patch features (frozen)", 16, patch_seconds, patch_held_out)):
    print(f"{label:34s} {tokens:6d} {seconds:8.2f} {report['colour']:16.3f} {report['size']:6.3f} {report['orientation']:12.3f}")
print(f"patch tokens, seen combinations: {patch_seen}")
assert patch_seen["all"] >= 0.75, "the patch-token model should still learn the seen combinations"

**Expected behaviour.** The patch-token model processes 24 positions instead of 9, so each step takes longer. It answers colour and orientation well, but its **size** answers are much worse, and on held-out combinations they are worse than a coin flip: about 0.43 in our run, and between 0.05 and 0.49 across eight seeds. Why? Every patch feature describes only its own 4 × 4 patch, and this tiny encoder never mixes patches, so no single token says how long the bar is. In the training data, however, the colour and orientation of a bar often predict its size: for example, every red vertical bar in training is large, because the small red vertical bar is held out. The model found that shortcut instead of the harder visual evidence. The aligned embedding, by contrast, was trained in notebook 2 to carry size for every colour, so the shortcut was not needed.

This is not a verdict against patch tokens: real encoders mix information across patches with self-attention, and spatial questions need patch-level detail. It is a demonstration that what the connector receives decides what the language model *can* use, and that held-out combinations reveal shortcuts that seen ones hide. Notebook 4 builds a direct test for this kind of shortcut.

## Exercises

1. **✍️ `mlp_projector`.** LLaVA-1.5 replaced the linear projector with a two-layer MLP. Write one below, then try it in place of `project_visual_tokens` by giving the projector dictionary the extra weights and editing `vqa_logits`. Does it change held-out accuracy in this small setting?
2. **A yes/no question.** Add "is the bar red ?" with answers `yes` and `no` to the vocabulary and the data. Generate it so that half of the answers are `yes`. What accuracy does a model reach that always answers `no`, and how far above that does yours get?
3. **Several prefix tokens.** Map the aligned embedding to $k = 4$ visual tokens instead of one, with a projector whose output has width $4 \times 24$ and a reshape. Does more visual context help when the information is the same?

### ✍️ Exercise 1: `mlp_projector`

Write `mlp_projector(features, W1, b1, W2, b2)`, returning $\operatorname{ReLU}(\mathbf{f}W_1 + \mathbf{b}_1)W_2 + \mathbf{b}_2$ for features of shape (n, N, d_vision), with $W_1$ of shape (d_vision, hidden) and $W_2$ of shape (hidden, d_model). LLaVA-1.5 uses a GELU activation; the engine has ReLU, which plays the same role here.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def mlp_projector_reference(features, W1, b1, W2, b2):
    """A two-layer projector: ReLU(features @ W1 + b1) @ W2 + b2. (n, N, d_vision) -> (n, N, d_model)."""
    return (features @ W1 + b1).relu() @ W2 + b2

In [ ]:
def mlp_projector(features, W1, b1, W2, b2):
    """A two-layer projector: ReLU(features @ W1 + b1) @ W2 + b2. (n, N, d_vision) -> (n, N, d_model)."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
case_rng = np.random.default_rng(36)
mlp_projector = practice_tensor("mlp_projector", mlp_projector, mlp_projector_reference, cases=[
    (Tensor(case_rng.normal(size=(2, 1, 16))), parameter(16, 32, rng=case_rng), parameter(32, rng=case_rng, scale=0.1),
     parameter(32, 24, rng=case_rng), parameter(24, rng=case_rng, scale=0.1)),
    (Tensor(case_rng.normal(size=(1, 3, 2))), parameter(2, 4, rng=case_rng), parameter(4, rng=case_rng, scale=0.1),
     parameter(4, 3, rng=case_rng), parameter(3, rng=case_rng, scale=0.1)),
])

## Common errors and debugging

- **Off-by-one answer positions.** The answer is predicted one position *before* it appears. Supervising position $N + q$ instead of $N + q - 1$ trains the model to predict `<end>` where the answer belongs; every answer then reads as `<end>`. Print the decoded targets at the masked positions, as Part 4 does.
- **Training on the prompt.** Without the mask, most of the loss comes from predicting the fixed question words, which teaches nothing about images and swamps the answer signal.
- **A missing causal mask.** If later positions are visible, the model can copy the answer from the input during training and then fails at test time, when the answer is not there.
- **"Frozen" weights that still move.** Excluding weights from the optimiser is not enough if something else updates them, and leaving `requires_grad=True` wastes computation on gradients nobody uses. Check, as the final cell does, that frozen weights are bit-for-bit unchanged.
- **Too large a step for pretrained weights.** Unfreezing an encoder with the same step size as freshly initialised layers can erase what pretraining taught it. Watch held-out accuracy, not only the training loss.
- **Answers that ignore the image.** High accuracy on a fixed question set can come from the text alone when answers are predictable from the question. Always run an image-swap or no-image control.

## Summary

- A connector maps vision features into the language model's embedding space; a linear projector is the simplest, and the language model then reads the visual tokens like word embeddings.
- The input is visual tokens followed by the question and answer; next-token targets are the input shifted by one, and an answer-only mask supervises just the answer and `<end>`.
- Freezing the vision encoder keeps its aligned features intact and trains fewer weights; unfreezing can help or hurt, depending on the data and the step size.
- An image-swap control showed that the answers depend on the image; patch tokens revealed a size shortcut that held-out combinations exposed.

## Next

In **notebook 4** you evaluate a vision-language model properly: accuracy by slice, counterfactual images that change one attribute at a time, hallucination on images with no bar, the token budget of images and video frames, and the capstone project.